# Проверка двух конфигураций RAG на 10 вопросах

Каждая конфигурация отвечает на одни и те же восемь вопросов по регламенту и два вопроса-ловушки.

| Конфигурация | Поиск | Контекст |
|---|---|---|
| YandexGPT 5 Lite 8B | BM25 и multilingual-e5-small, объединение RRF, переранжирование | 5 фрагментов |
| Gemma 3 4B | BM25 по полю `embed_text` | 3 фрагмента |

В Colab выберите среду с GPU и выполните ячейки до скачивания CSV. После разметки вернитесь в раздел 4. Там можно открыть готовый файл даже в новом сеансе, без повторной генерации.

## 1. Подготовка окружения

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd()
if IN_COLAB:
    ROOT = Path("/content/gasoline-blending-rag")
    if not (ROOT / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/stwz13/gasoline-blending-rag.git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank_bm25", "pymorphy3",
                    "faiss-cpu", "sentence-transformers", "nltk", "pydantic>=2", "requests", "httpx", "pandas"], check=True)
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Папка проекта:", ROOT)

In [ ]:
import hashlib
import json
import shutil
import time
from unittest.mock import patch

import pandas as pd
import requests

from src.evaluate import load_queries, recall_at_k
from src.generate import NOT_FOUND, build_context, build_messages
from src.hybrid_retrieval import build_hybrid
from src.retrieval import default_bm25
from src.settings import load_settings

models = {"default": "hf.co/yandex/YandexGPT-5-Lite-8B-instruct-GGUF", "local": "gemma3:4b"}
configs = {}
for profile, model in models.items():
    with patch.dict(os.environ, {"RAG_PROFILE": profile, "RAG_MODEL": model}):
        configs[profile] = load_settings()

queries = load_queries(ROOT / "data/test_queries.jsonl")
assert len(queries) == len({q["query_id"] for q in queries}) == 10
assert sum(q["category"] == "trap" for q in queries) == 2
pd.DataFrame([{"профиль": profile, "модель": cfg.ollama.model, "поиск": cfg.retrieval.method,
               "переранжирование": cfg.retrieval.rerank, "фрагментов": cfg.retrieval.default_k,
               "окно, токенов": cfg.ollama.num_ctx} for profile, cfg in configs.items()])

In [ ]:
base_url = configs["default"].ollama.base_url.rstrip("/")
if IN_COLAB:
    if not shutil.which("ollama"):
        if not shutil.which("zstd"):
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
        subprocess.run(["bash", "-o", "pipefail", "-c", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
    try:
        requests.get(f"{base_url}/api/tags", timeout=2).raise_for_status()
    except requests.ConnectionError:
        with open("/tmp/ollama.log", "w") as log:
            server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)
        for _ in range(30):
            try:
                requests.get(f"{base_url}/api/tags", timeout=2).raise_for_status()
                break
            except requests.ConnectionError:
                time.sleep(1)
        else:
            raise RuntimeError(Path("/tmp/ollama.log").read_text()[-2000:])

tags = requests.get(f"{base_url}/api/tags", timeout=10)
tags.raise_for_status()
digests = {m["name"].removesuffix(":latest").casefold(): m["digest"] for m in tags.json()["models"]}
for model in models.values():
    if model.casefold() not in digests:
        print("Загрузка", model, flush=True)
        subprocess.run(["ollama", "pull", model], check=True)
tags = requests.get(f"{base_url}/api/tags", timeout=10)
tags.raise_for_status()
digests = {m["name"].removesuffix(":latest").casefold(): m["digest"] for m in tags.json()["models"]}
print("Готовы к прогону:", *models.values(), sep="\n")

## 2. Прогон моделей

Поиск и генерация выполняются последовательно. После каждого ответа обновляется общий CSV. При повторном запуске ячейки готовые пары «профиль, вопрос» пропускаются. Эталоны и выдержки для разметки сохраняются в CSV отдельно от запроса к модели.

In [ ]:
signature = hashlib.sha256()
for path in ["config.toml", "data/test_queries.jsonl", "data/chunks.jsonl", "src/generate.py",
             "src/hybrid_retrieval.py", "src/retrieval.py", "src/evaluate.py", "src/settings.py"]:
    signature.update((ROOT / path).read_bytes())
for profile, model in models.items():
    signature.update(f"{profile}|{model}|{digests[model.casefold()]}".encode())

run_id = signature.hexdigest()[:16]
OUT = ROOT / "data/generation" / f"holdout_two_models_{run_id}.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
print(OUT)

In [ ]:
def make_retriever(config):
    search = config.retrieval
    if search.method == "bm25" and not search.rerank:
        return default_bm25(field=search.bm25_field)
    if search.method in {"rrf", "weighted"}:
        return build_hybrid(search.method, search.encoder, search.alpha, search.rrf_k,
                            search.rerank, bm25_field=search.bm25_field, rerank_depth=search.rerank_depth)
    raise ValueError(f"Неизвестная конфигурация поиска: {search}")


def run_question(profile, config, retriever, q):
    start = time.perf_counter()
    hits = retriever.search(q["query"], k=config.retrieval.default_k, expand=False)
    search_seconds = time.perf_counter() - start
    llm = config.ollama
    options = llm.model_dump(include={"temperature", "num_ctx", "num_thread", "num_predict"}, exclude_none=True)
    payload = {"model": llm.model, "messages": build_messages(q["query"], hits),
               "stream": False, "options": options}
    if llm.keep_alive is not None:
        payload["keep_alive"] = llm.keep_alive

    start = time.perf_counter()
    response = requests.post(f"{llm.base_url.rstrip('/')}/api/chat", json=payload, timeout=llm.timeout)
    response.raise_for_status()
    answer = response.json()["message"]["content"].strip()
    seconds = time.perf_counter() - start
    if not answer:
        raise ValueError(f"Пустой ответ на вопрос {q['query_id']}")
    texts = [hit.chunk["text"] for hit in hits]
    return {
        "run_id": run_id, "profile": profile, "model": llm.model, "model_digest": digests[llm.model.casefold()],
        "query_id": q["query_id"], "category": q["category"], "question": q["query"],
        "reference": q["reference"], "evidence": "\n".join(q["evidence"]), "answer": answer,
        "sources": build_context(hits), "ranked_ids": " ".join(hit.chunk_id for hit in hits),
        "k": config.retrieval.default_k, "hit_in_context": None if q["category"] == "trap" else
            recall_at_k(texts, q["evidence"], k=len(hits)),
        "exact_refusal": int(answer == NOT_FOUND), "search_seconds": round(search_seconds, 3),
        "generation_seconds": round(seconds, 2), "accuracy": "", "hallucination": "", "note": "",
    }

In [ ]:
def run():
    rows = pd.read_csv(OUT, keep_default_na=False).to_dict("records") if OUT.exists() else []
    expected = {(profile, q["query_id"]) for profile in configs for q in queries}
    done = {(row["profile"], row["query_id"]) for row in rows}
    if len(rows) != len(done) or not done <= expected or any(row["run_id"] != run_id for row in rows):
        raise ValueError("Проверьте файл предыдущего прогона.")

    for profile, cfg in configs.items():
        todo = [q for q in queries if (profile, q["query_id"]) not in done]
        if not todo:
            continue
        print(f"{cfg.ollama.model}: осталось вопросов {len(todo)}", flush=True)
        retriever = make_retriever(cfg)
        for q in todo:
            row = run_question(profile, cfg, retriever, q)
            rows.append(row)
            tmp = OUT.with_suffix(".tmp")
            pd.DataFrame(rows).to_csv(tmp, index=False, encoding="utf-8-sig")
            tmp.replace(OUT)
            print(f"{len(rows)}/20  {profile}  {q['query_id']}  {row['generation_seconds']} с", flush=True)
        del retriever
    print(f"Сохранено ответов: {len(rows)} из 20")


run()

## 3. Ответы для разметки

Скачайте один CSV с двадцатью ответами. Для каждой строки сравните ответ с эталоном и источниками. В `accuracy` поставьте 2 за полный верный ответ, 1 за частичный, 0 за неверный ответ или ложный отказ. Обоснованный отказ на ловушке получает 2. В `hallucination` поставьте 1, если есть неподтверждённое утверждение, иначе 0. В `note` укажите причину спорной оценки.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT))
else:
    print("Файл для разметки:", OUT)

## 4. Сравнение конфигураций

При первом прогоне таблица покажет результаты поиска, точные отказы и время генерации. Когда CSV будет размечен, установите `UPLOAD_MARKED = True` и повторите две ячейки ниже. В Colab выберите размеченный файл. Эти ячейки можно выполнить и в свежем сеансе. Для локального запуска укажите путь к размеченному CSV в `RESULT_FILE`.

In [ ]:
from pathlib import Path
import io
import pandas as pd

UPLOAD_MARKED = False  # @param {type:"boolean"}
RESULT_FILE = ""  # путь к готовому CSV при локальном запуске

if UPLOAD_MARKED:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Выберите один CSV с ответами двух конфигураций.")
    source = io.BytesIO(next(iter(uploaded.values())))
elif RESULT_FILE:
    source = Path(RESULT_FILE)
else:
    source = OUT

answers = pd.read_csv(source, encoding="utf-8-sig", keep_default_na=False)
keys = ["profile", "query_id"]
groups = answers.groupby("profile")
if (answers["run_id"].nunique() != 1 or answers.duplicated(keys).any()
        or groups.size().to_dict() != {"default": 10, "local": 10}
        or answers.groupby("query_id").size().ne(2).any()
        or answers.groupby("query_id")[["question", "category", "reference"]].nunique().ne(1).any().any()
        or groups["model"].nunique().ne(1).any() or answers["model"].nunique() != 2):
    raise ValueError("Для сравнения нужны ответы обеих конфигураций на одни и те же 10 вопросов.")
for column, allowed in {"accuracy": [0, 1, 2], "hallucination": [0, 1]}.items():
    answers[column] = pd.to_numeric(answers[column].replace("", float("nan")), errors="raise")
    if not answers[column].dropna().isin(allowed).all():
        raise ValueError(f"Проверьте значения в {column}.")
print("Размечено ответов:", len(answers.dropna(subset=["accuracy", "hallucination"])), "из 20")

In [ ]:
def metrics(df):
    regular = df[df.category != "trap"]
    traps = df[df.category == "trap"]
    result = {
        "ответов": len(df), "фрагментов в контексте": df.k.iloc[0],
        "эталон найден в контексте (из 8)": pd.to_numeric(regular.hit_in_context).sum(),
        "точных отказов на ловушках (из 2)": traps.exact_refusal.sum(),
        "медиана поиска, с": df.search_seconds.median(),
        "медиана генерации, с": df.generation_seconds.median(),
    }
    if answers[["accuracy", "hallucination"]].notna().all().all():
        result.update({
            "полностью верных (из 10)": df.accuracy.eq(2).sum(),
            "средний балл (0–2)": df.accuracy.mean(),
            "ответов с галлюцинациями (из 10)": df.hallucination.sum(),
            "ловушек распознано по смыслу (из 2)": traps.accuracy.eq(2).sum(),
        })
    return pd.Series(result)

display(pd.DataFrame({model: metrics(df) for model, df in answers.groupby("model", sort=False)}).round(3))

errors = answers[(answers.accuracy < 2) | (answers.hallucination == 1)]
if not errors.empty:
    display(errors[["query_id", "profile", "question", "answer", "accuracy", "hallucination", "note"]])

## Вывод по тесту

Заполните после разметки всех 20 ответов.

На десяти тестовых вопросах конфигурация с YandexGPT дала ___ полностью верных ответов, а конфигурация с Gemma — ___. Из двух ловушек они распознали ___ и ___ соответственно. Пример различия на одном и том же вопросе — ___.

Медианное время генерации в этой среде составило ___ с у YandexGPT и ___ с у Gemma. По совокупности качества и времени для ___ я бы выбрала ___, потому что ___.

В этом опыте одновременно менялись модель, поиск и число фрагментов. Сравнение характеризует две конфигурации целиком.